# (APPLY/ASSESS) GRADED ASSIGNMENT (PROGRAMMING) | Multi-Source Retrieval Pipeline (45 min)

**Assignment type:** Graded Assignment (Programming)
**Tools:** Azure Databricks cluster

## Overview
You will build and evaluate a production-grade, multi-source retrieval pipeline within an Azure Databricks notebook. This hands-on project requires you to orchestrate three distinct candidate generation models (ALS, Two-Tower, and LightGCN), blend their outputs, and evaluate whether the combined system provides a recall benefit compared to any single source.

## Goals
By completing this activity, you will:
- Construct an integrated multi-source retrieval system blending candidates from collaborative filtering, deep neural, and graph-based models.
- Implement parallel candidate generation using Python's `concurrent.futures`, applied only where it's safe to run concurrently.
- Apply global string ID deduplication to create a unified, clean candidate pool.
- Evaluate a combined candidate pool using the Recall@100 metric and compare it against each individual source.

## Scenario
You are a Lead Machine Learning Engineer at a large enterprise platform, tasked with improving the quality and diversity of your recommendation engine's retrieval stage. Your existing retrieval model fails to capture niche user interests and struggles with the cold-start problem. Your objective is to architect a new solution that combines the strengths of three different models: an ALS collaborative filtering model, a Two-Tower neural network, and a LightGCN graph-based model.

A latency-conscious design is part of the engineering brief for this pipeline — production retrieval systems are typically expected to respond quickly, which is why this task emphasizes parallel orchestration wherever it's safe to run concurrently. Treat any specific latency target as a design goal to discuss in your architecture document, not a number this notebook measures or enforces.

Your task is to write a master Python script that generates candidates from all three sources — running the non-Spark sources in parallel — merges their outputs into a single deduplicated pool, and evaluates the final combined candidate list against a Recall@100 benchmark.


---
## Task 1: Initialize the Orchestration Environment (10 min)

**Context:** Before you can orchestrate the models, you must load all necessary libraries and pre-computed artifacts into your Azure Databricks notebook. These artifacts are stored in a **Unity Catalog Volume**, not the legacy Databricks File System (DBFS) — Volumes are required here because Spark model artifacts like the ALS model involve distributed reads/writes across executor tasks, which the legacy DBFS/Workspace filesystem does not support.

**Practice:**
- Complete the code block below by loading each artifact from its Volume path into a variable.
- After loading each artifact, use a new cell to inspect it (`.shape`, `.head()`, `spark.version`).
- Run a local validation check on the loaded artifacts before moving on — confirm dimensions and expected columns match what you expect.

**Consider:** Why is it important to verify the dimensions of the loaded `two_tower_embeddings` array before proceeding? What kind of error could a dimension mismatch cause in the next task?

**Success Checklist:**
- [ ] The ALS model is successfully loaded into a variable using `ALSModel.load()`.
- [ ] The Two-Tower embeddings and item IDs are loaded into NumPy arrays.
- [ ] The LightGCN candidates and ground truth data are loaded into pandas DataFrames.
- [ ] A local validation cell confirms array/DataFrame shapes and schema before proceeding.
- [ ] All notebook cells execute without file-not-found or deserialization errors.


In [ ]:
import concurrent.futures
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALSModel

# Paths to pre-computed artifacts in a Unity Catalog Volume
CATALOG_NAME = "courses_workspace"
SCHEMA_NAME = "default"
VOLUME_NAME = "lab_assets_vol"
ASSETS_DIR = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

ALS_MODEL_PATH = f"{ASSETS_DIR}/als_model"
TWO_TOWER_EMBEDDINGS_PATH = f"{ASSETS_DIR}/two_tower_item_embeddings.npy"
TWO_TOWER_IDS_PATH = f"{ASSETS_DIR}/two_tower_item_ids.npy"
LIGHTGCN_CANDIDATES_PATH = f"{ASSETS_DIR}/lightgcn_candidates.parquet"
GROUND_TRUTH_PATH = f"{ASSETS_DIR}/ground_truth_interactions.parquet"

# --- TODO: Your code below to load the artifacts ---

# Load the ALS model
als_model = None  # TODO: replace with ALSModel.load(...)

# Load the Two-Tower embeddings and IDs
two_tower_embeddings = None  # TODO
two_tower_item_ids = None    # TODO

# Load the LightGCN candidates
lightgcn_candidates = None  # TODO

# Load the ground truth evaluation data
ground_truth = None  # TODO


In [ ]:
# Inspect the loaded artifacts
print("Spark version:", spark.version)

# TODO: print two_tower_embeddings.shape
# TODO: print len(two_tower_item_ids)
# TODO: print lightgcn_candidates.head()
# TODO: print ground_truth.head()


In [ ]:
# Local validation checks — confirm dimensions and schema before moving on
# TODO: assert two_tower_embeddings.shape[0] == len(two_tower_item_ids), "Embedding/ID count mismatch"
# TODO: assert "user_id" in lightgcn_candidates.columns and "item_id" in lightgcn_candidates.columns
# TODO: assert "user_id" in ground_truth.columns and "item_id" in ground_truth.columns

# If every assert above passes, execution reaches this line without error.
print("All validation checks passed.")


---
## Task 2: Implement Candidate Generation (15 min)

**Context:** You'll generate candidates from all three sources. The ALS lookup depends on the active SparkContext, and Spark operations are **not safe to call from inside a `ThreadPoolExecutor` worker thread** under Databricks' pinned-thread execution model. So ALS runs on the main thread, and only the two NumPy/pandas-based sources — Two-Tower and LightGCN — run concurrently. This still gives you a real parallelization benefit without the cross-thread Spark issue.

**Practice:**
- Complete `generate_als_candidates` by ranking each user's candidate items via a dot product between their factor vector and every item's factor vector, then taking the top-k.
- Call `generate_als_candidates` directly (main thread) before entering the executor block.
- Instantiate a `concurrent.futures.ThreadPoolExecutor` with `max_workers=2`.
- Within the `with` block, submit `generate_two_tower_candidates` and `generate_lightgcn_candidates` to the executor.
- Collect each function's output into a separate results variable (`als_results`, `tt_results`, `lgcn_results`).

**Consider:** Why does ALS need to stay on the main thread here, while Two-Tower and LightGCN don't? What's different about what each function actually does under the hood?

**Success Checklist:**
- [ ] `generate_als_candidates` produces ranked candidates from the model's raw factors, without calling `recommendForUserSubset`.
- [ ] `ThreadPoolExecutor` is used to orchestrate the two non-Spark generation functions.
- [ ] The `submit` method is called for each of those two functions.
- [ ] The `.result()` method is called for each future to retrieve its output.
- [ ] The code block executes without errors and produces candidate lists from all three sources.


In [ ]:
# Sample user inputs
# NOTE: user_id is an INTEGER throughout this pipeline; item_id is a STRING ("item_N").
# Keeping this consistent avoids type-mismatch bugs when merging candidates later.
user_ids = ground_truth["user_id"].unique().tolist()
embedding_dim = two_tower_embeddings.shape[1]
rng = np.random.default_rng(seed=42)
user_embeddings = rng.normal(size=(len(user_ids), embedding_dim))


In [ ]:
# --- Starter functions for each generation source ---

def generate_als_candidates(user_ids, top_k=100):
    als_model = ALSModel.load(ALS_MODEL_PATH)

    # Read the model's learned factors directly rather than calling
    # als_model.recommendForUserSubset(), which relies on Spark
    # higher-order functions not supported on this cluster type.
    user_factors_df = als_model.userFactors.toPandas()
    item_factors_df = als_model.itemFactors.toPandas()

    # --- TODO: Your code below ---
    # 1. Stack user_factors_df["features"] and item_factors_df["features"] into NumPy matrices
    # 2. Build a user_id -> row index lookup from user_factors_df["id"]
    # 3. For each requested uid, compute a dot product against all item factors,
    #    take the top_k indices (descending), and record a row:
    #    {"user_id": uid, "item_id": f"item_{item_idx}"}
    # 4. Return a pandas DataFrame of the collected rows
    pass


def generate_two_tower_candidates(user_embeddings, top_k=100):
    scores = np.dot(user_embeddings, two_tower_embeddings.T)
    # Sort descending so the top similarity scores come first
    top_indices = np.argsort(scores, axis=1)[:, ::-1][:, :top_k]
    return two_tower_item_ids[top_indices]


def generate_lightgcn_candidates(user_ids, top_k=100):
    return lightgcn_candidates[lightgcn_candidates["user_id"].isin(user_ids)
    ].groupby("user_id").head(top_k)


In [ ]:
# --- TODO: Your code below to execute candidate generation ---

# 1. Run generate_als_candidates on the MAIN THREAD (not inside the executor)
als_results = None  # TODO

# 2. Use a ThreadPoolExecutor (max_workers=2) to run Two-Tower and LightGCN concurrently
tt_results = None    # TODO
lgcn_results = None  # TODO

# print(f"ALS returned {len(als_results)} recommendations.")
# print(f"Two-Tower returned candidates for {len(tt_results)} users.")
# print(f"LightGCN returned {len(lgcn_results)} candidate rows.")


---
## Task 3: Blend, Deduplicate, and Evaluate (20 min)

**Context:** With candidates from all three sources, your final task is to merge them into a single pool, handle duplicates, and evaluate the result. The most critical steps are ensuring all item identifiers are consistent string types and that deduplication happens globally, across all three sources at once.

**Practice:**
- Convert the item IDs from each of the three results into a Python list of strings.
- Use the `|` (union) operator to combine the three sets into one final deduplicated set.
- **Note:** the per-source lists preserve rank order, but a Python `set` does not — when evaluating the combined pool, decide whether truncating to top-100 is meaningful for an unordered set, or whether the combined pool should be evaluated in full.
- Complete the `recall_at_k` function: intersection of candidate IDs and ground truth IDs, divided by the total number of ground truth items.
- Print Recall@100 for ALS, Two-Tower, LightGCN, and the combined pool.
- Analyze your output: Does the combined Recall show a lift over the best individual source? If not, what could be wrong — or is the combined pool being truncated in a way that doesn't reflect its actual contents?

**Success Checklist:**
- [ ] All item IDs are correctly cast to strings before being added to sets.
- [ ] A set union is used to perform a global deduplication across all three sources.
- [ ] The `recall_at_k` function correctly calculates the metric.
- [ ] You report and briefly justify the Combined Recall result — a numeric lift over every individual source is the expected typical outcome with this lab's data, but your writeup should explain your result rather than assume it.


In [ ]:
# --- TODO: Your code below to normalize, blend, and evaluate ---

# 1. Normalize all candidate item IDs from each source to a consistent string format
als_ids = None   # TODO
tt_ids = None    # TODO
lgcn_ids = None  # TODO

# 2. Use the set union operator to merge the three sets into a single, deduplicated set
combined_candidate_ids = None       # TODO: set(als_ids) | set(tt_ids) | set(lgcn_ids)
combined_candidate_list = None      # TODO: list(combined_candidate_ids)


# 3. Implement the recall@k function
def recall_at_k(candidate_ids, ground_truth_ids_set, k=100):
    # TODO: Complete this function.
    # - Convert candidate_ids to a list
    # - If k is not None, truncate to the first k items
    # - Compute the intersection with ground_truth_ids_set
    # - Return hits / len(ground_truth_ids_set), guarding for an empty ground truth set
    pass


# 4. Calculate and print Recall@100 for each source AND the combined pool
gt_set = None  # TODO: set(ground_truth["item_id"].astype(str))

# als_recall = recall_at_k(als_ids, gt_set, k=100)
# tt_recall = recall_at_k(tt_ids, gt_set, k=100)
# lgcn_recall = recall_at_k(lgcn_ids, gt_set, k=100)
# combined_recall = recall_at_k(combined_candidate_list, gt_set, k=None)  # full pool, not truncated

# print(f"ALS Recall@100: {als_recall:.4f}")
# print(f"Two-Tower Recall@100: {tt_recall:.4f}")
# print(f"LightGCN Recall@100: {lgcn_recall:.4f}")
# print(f"Combined Recall (full deduplicated pool): {combined_recall:.4f}")


### Common Mistakes to Avoid
- **Running ALS from a worker thread**: calling `generate_als_candidates` from inside the `ThreadPoolExecutor` alongside the other two functions will fail under Databricks' pinned-thread execution model. Keep it on the main thread.
- **Using `recommendForUserSubset` on a Unity Catalog Shared-access-mode cluster**: this method depends on Spark higher-order functions that this cluster type does not support, and will fail with a `UC_COMMAND_NOT_SUPPORTED` error.
- **Inconsistent ID Types**: forgetting to cast all item identifiers to a string format before deduplication. A set will treat the integer `123` and the string `'123'` as two distinct items.
- **Source-Level Deduplication**: deduplicating within each source's candidate list before merging misses cross-source duplicates.
- **Incorrect Recall Calculation**: calculating recall based on raw hit count without normalizing by the ground truth set size.
- **Truncating an unordered set at `[:100]`**: slicing the combined `set` after union does not give you a meaningful "top 100," since sets have no defined order.

### Key Points
- Multi-source blending is a production-standard technique for increasing recall and ensuring coverage across diverse user segments and item types.
- Parallelization should be applied where it's safe — I/O-bound, non-Spark operations are good candidates for a thread pool; Spark-dependent operations are not safe to run from arbitrary worker threads under Databricks' pinned-thread execution model.
- Global string ID deduplication is the required method for creating a clean, unified candidate pool from multiple sources.


---
## Submission Prompts

**Prompt 1:** Submit your fully executed Azure Databricks Jupyter Notebook file (`.ipynb`). The autograder will evaluate the notebook for successful execution, correct generation logic, and the final printed Combined Recall result.

**Prompt 2:** Submit your portfolio-ready technical architecture document in Markdown format (`.md`). This document should briefly explain the design of your pipeline, your deduplication strategy, why ALS runs on the main thread while the other two sources run in a thread pool, and a summary of your final recall scores.

**Prompt 3:** In 2–3 sentences, explain why using a Python `set` and the union operator (`|`) is a more computationally efficient method for global deduplication than combining lists and iterating through them to find unique items.
